# 9차시 실습 — 에이전트에 '도구'를 더한다 (Tool Use & MCP)

> **day1·** 7차시에서 만든 **맛집 추천 에이전트(ReAct 루프)** 에, 오늘은 **함수 도구 2개**(API류 + 로컬)를 더 붙입니다.
> 마지막엔 **MCP 등록**(`codex mcp add …`)으로 표준 연결을 보고, **내 팀 MVP의 도구**로 그대로 적용합니다.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 실행하세요.
- 7차시 노트북을 이어서 쓰는 흐름입니다(환경 셀은 동일).

In [1]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

7차시: 맛집 추천 에이전트가 `search_restaurants`·`budget_check` 두 도구를 **스스로 골라** 답했습니다.
9차시: 같은 에이전트에 **도구를 더 단다** — 모델이 약한 일을 도구로 보완합니다.

- **`web_search`** — 외부 정보(리뷰·영업시간)를 가져오는 **API류 도구**(여기선 정해진 결과를 주는 **스텁**, 실제 API 자리).
- **`calculator`** — 수식을 정확히 계산하는 **로컬 도구**(LLM은 큰 수 계산을 틀려도 코드는 안 틀림).

> 구조는 7차시와 동일 — **도구만 추가**하면 됩니다. 루프(ReAct)는 그대로 재사용.

## STEP 1 — 7차시 도구(베이스) 다시 정의

먼저 7차시의 도구와 ReAct 루프를 그대로 가져옵니다. (여기에 STEP 2에서 도구를 *추가*합니다.)

⌨️ **터미널 Codex:** `codex \"7차시 맛집 추천 에이전트에 도구를 추가할 거야. 기존 도구·루프부터 보여줘\"`

In [21]:
# 7차시 베이스 — 가짜 데이터 + 도구 2개 (그대로)
RESTAURANTS = {
  "강남": [{"name":"매운갈비찜집","price":18000,"taste":"매움"},
          {"name":"순한국밥","price":9000,"taste":"순함"},
          {"name":"불막창","price":22000,"taste":"매움"}],
  "홍대": [{"name":"치즈파스타","price":15000,"taste":"순함"}],
}
def search_restaurants(area: str, taste: str = "") -> str:
    items = RESTAURANTS.get(area, [])
    if taste: items = [r for r in items if r["taste"]==taste]
    return json.dumps(items, ensure_ascii=False) if items else "결과 없음"
def budget_check(price: int, people: int) -> str:
    return f"1인당 {price//max(people,1):,}원"

TOOL_IMPL = {"search_restaurants": search_restaurants, "budget_check": budget_check}
TOOLS = [
 {"type":"function","function":{"name":"search_restaurants","description":"지역(과 맛)으로 맛집 검색",
   "parameters":{"type":"object","properties":{"area":{"type":"string"},"taste":{"type":"string","description":"매움/순함, 선택"}},"required":["area"]}}},
 {"type":"function","function":{"name":"budget_check","description":"총가격과 인원으로 1인당 가격 계산",
   "parameters":{"type":"object","properties":{"price":{"type":"integer"},"people":{"type":"integer"}},"required":["price","people"]}}},
]
print("베이스 도구:", list(TOOL_IMPL))

베이스 도구: ['search_restaurants', 'budget_check']


## STEP 2 — ⭐ 도구 2개 추가: `web_search`(API류) + `calculator`(로컬)

오늘의 핵심. 모델은 글은 잘 쓰지만 **최신 정보·정확한 계산**은 약합니다 → **도구로 보완**합니다.
- 메타데이터가 중요합니다: **이름은 좁게, 설명은 명확·구별되게, 스키마는 엄격하게.**

⌨️ **터미널 Codex:** `codex \"맛집 에이전트에 web_search(스텁)와 calculator 도구를 스키마까지 추가해줘\"`

In [22]:
# 추가 도구 ① web_search — API류 도구 (여기선 정해진 결과만 주는 '스텁' = 실제 API 자리)
FAKE_WEB = {
  "매운갈비찜집 리뷰": "평점 4.5/5, '매콤하고 양 많음', 평일 점심 웨이팅 10분",
  "불막창 영업시간": "매일 16:00~02:00, 점심 영업 안 함",
}
def web_search(query: str) -> str:
    return FAKE_WEB.get(query, "검색 결과 없음(스텁) — 실제로는 외부 검색 API 호출")

# 추가 도구 ② calculator — 로컬 도구 (정밀·예측가능)
def calculator(expr: str) -> str:
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"계산 오류: {e}"

# 기존 TOOL_IMPL / TOOLS 에 '추가'만 한다 — 루프는 그대로
TOOL_IMPL.update({"web_search": web_search, "calculator": calculator})
TOOLS += [
 {"type":"function","function":{"name":"web_search","description":"맛집 리뷰·영업시간 등 최신 정보를 외부에서 검색(API류)",
   "parameters":{"type":"object","properties":{"query":{"type":"string","description":"검색어"}},"required":["query"]}}},
 {"type":"function","function":{"name":"calculator","description":"수식 문자열을 정확히 계산(로컬, 산술 전용)",
   "parameters":{"type":"object","properties":{"expr":{"type":"string","description":"예: '18000*2*0.9'"}},"required":["expr"]}}},
]
print("도구 4개로 확장:", list(TOOL_IMPL))

도구 4개로 확장: ['search_restaurants', 'budget_check', 'web_search', 'calculator']


## STEP 3 — ReAct 루프 실행: 4개 도구 중 스스로 고르기

루프 코드는 **7차시와 동일**합니다(도구만 늘었을 뿐). 모델이 어떤 도구를 언제 부르는지 관찰하세요.

In [23]:
def run_agent(question, max_steps=6, verbose=True):
    messages=[{"role":"system","content":"너는 맛집 추천 도우미다. 검색·계산·웹검색 도구를 필요할 때만 쓰고, 충분하면 한국어로 추천 이유와 함께 답하라."},
              {"role":"user","content":question}]
    for step in range(1, max_steps+1):
        msg = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS).choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            if verbose: print(f"\n✅ [최종 답]\n{msg.content}")
            return msg.content
        for tc in msg.tool_calls:
            args=json.loads(tc.function.arguments); result=TOOL_IMPL[tc.function.name](**args)
            if verbose: print(f"🧠 step{step} 도구: {tc.function.name}({args}) → 👀 {result}")
            messages.append({"role":"tool","tool_call_id":tc.id,"content":str(result)})
    return "최대 스텝 도달"

# 검색 + 계산(로컬) + 웹검색(API류)을 한 요청 안에서 섞어 써본다
run_agent("강남에서 매운 집 추천하고, 2명이 10% 할인받으면 1인당 얼마야? 그 집 리뷰도 알려줘.")

🧠 step1 도구: search_restaurants({'area': '강남', 'taste': '매운'}) → 👀 결과 없음
🧠 step1 도구: budget_check({'price': 18000, 'people': 2}) → 👀 1인당 9,000원
🧠 step1 도구: web_search({'query': '강남 매운 맛집 리뷰'}) → 👀 검색 결과 없음(스텁) — 실제로는 외부 검색 API 호출

✅ [최종 답]
강남에서 매운 음식점을 찾는 데에는 정보가 부족한 것 같습니다. 대신, 저렴한 가격으로 개인당 9,000원으로 계산되었습니다. 

매운 맛집에 대한 리뷰도 확인할 수 없었습니다. 강남 지역에는 다양한 매운 맛집이 있으니, 인기 있는 가게들을 방문해 보시는 것도 좋습니다. 예를 들어, '매운탕'이나 '양념치킨' 전문점들이 많이 있기 때문에 방문해 보시길 추천합니다.

혹시 다른 지역이나 음식 종류에 대한 정보가 필요하시다면 말씀해 주세요!


"강남에서 매운 음식점을 찾는 데에는 정보가 부족한 것 같습니다. 대신, 저렴한 가격으로 개인당 9,000원으로 계산되었습니다. \n\n매운 맛집에 대한 리뷰도 확인할 수 없었습니다. 강남 지역에는 다양한 매운 맛집이 있으니, 인기 있는 가게들을 방문해 보시는 것도 좋습니다. 예를 들어, '매운탕'이나 '양념치킨' 전문점들이 많이 있기 때문에 방문해 보시길 추천합니다.\n\n혹시 다른 지역이나 음식 종류에 대한 정보가 필요하시다면 말씀해 주세요!"

## STEP 4 — iterate: 어떤 도구를 고르는지 바꿔 관찰

day1의 \"작게 요청 → 확인 → 수정\" 리듬 그대로. 요청에 따라 모델이 **다른 도구**를 부르는지 보세요.
(도구가 늘수록 선택이 흔들리면 → 강의의 **Less is More**: 도구를 줄이고 설명을 또렷이.)

In [5]:
run_agent("불막창은 점심에 가도 돼? 22000원짜리 3명이면 1인당 얼마인지도 계산해줘.")

🧠 step1 도구: budget_check({'price': 22000, 'people': 3}) → 👀 1인당 7,333원
🧠 step1 도구: web_search({'query': '불막창 점심 영업 시간'}) → 👀 검색 결과 없음(스텁) — 실제로는 외부 검색 API 호출

✅ [최종 답]
불막창은 점심에도 가셔도 좋습니다. 가격 계산 결과, 22,000원을 3명에서 나누면 1인당 7,333원입니다. 

가게의 영업시간은 필요한 경우에 따라 달라질 수 있으니, 방문 전에 확인하는 것이 좋습니다.


'불막창은 점심에도 가셔도 좋습니다. 가격 계산 결과, 22,000원을 3명에서 나누면 1인당 7,333원입니다. \n\n가게의 영업시간은 필요한 경우에 따라 달라질 수 있으니, 방문 전에 확인하는 것이 좋습니다.'

## STEP 5 — MCP 등록 읽기 (설명) — 표준으로 외부 도구 붙이기

함수 도구는 **내 코드 안 함수**입니다. 팀이 공유하거나 외부 시스템(문서·브라우저·GitHub)을 쓰려면 **MCP**(표준 포트)로 붙입니다.

- 함수 도구 3단계(정의→호출→실행)의 **표준판**: **등록 → 권한 확인(`/mcp`) → 호출**.
- 아래는 **실행이 아니라 설명**입니다. 터미널에서 Codex CLI에 한 줄로 등록합니다.

```bash
# 문서 검색 MCP(Context7) 등록
codex mcp add context7 -- npx -y @upstash/context7-mcp

# 브라우저 자동화 MCP(Playwright) 등록
codex mcp add playwright -- npx -y @playwright/mcp@latest

# 등록 확인 / 권한 점검
codex mcp list
```

> 위 `web_search` 스텁을 실제로 만들려면, 직접 API를 붙이는 대신 **검색 MCP 서버를 `codex mcp add`로 꽂는 것**이 표준적인 방법입니다.

⌨️ **터미널 Codex:** `codex mcp add context7 -- npx -y @upstash/context7-mcp` → 이어서 `/mcp` 로 연결 확인

In [6]:
# (선택) 등록된 MCP 서버 확인 — 터미널에서 실행하는 명령을 노트북에서 참고용으로
print("터미널에서 실행:")
print("  codex mcp add context7 -- npx -y @upstash/context7-mcp")
print("  codex mcp list   # 등록 확인")
print("  /mcp             # Codex 세션 안에서 연결·권한 점검")
print()
print("함수 도구 = 내 코드 안 함수 / MCP = 표준 포트로 꽂는 외부 서버")

터미널에서 실행:
  codex mcp add context7 -- npx -y @upstash/context7-mcp
  codex mcp list   # 등록 확인
  /mcp             # Codex 세션 안에서 연결·권한 점검

함수 도구 = 내 코드 안 함수 / MCP = 표준 포트로 꽂는 외부 서버


## STEP 6 — 메타데이터 실험: 설명을 흐리면 선택이 무너진다

강의의 "메타데이터가 로직만큼 중요하다"를 직접 확인합니다. `calculator`의 설명을 모호하게 바꾸면 모델이 도구를 잘못 고르거나 안 고릅니다.

In [26]:
import copy
TOOLS_GOOD = copy.deepcopy(TOOLS)          # 원본 백업

# 설명을 일부러 모호하게 — 이름도 일반적으로
for t in TOOLS:
    if t["function"]["name"] == "calculator":
        t["function"]["name"] = "google_search"
        t["function"]["description"] = "웹을 검색한다"
TOOL_IMPL["google_search"] = TOOL_IMPL["calculator"]

print("=== 나쁜 메타데이터: '1000/3=?' ===")
run_agent("1000/3=?", max_steps=3)

TOOLS[:] = TOOLS_GOOD                       # 복구
TOOL_IMPL.pop("google_search", None)
print("\n=== 복구 후 (좋은 메타데이터) ===")
run_agent("1000/3=?", max_steps=3)
# 관찰: 모델은 코드를 못 본다 — 이름·설명·스키마가 선택의 전부

=== 나쁜 메타데이터: '1000/3=?' ===

✅ [최종 답]
1000을 3으로 나누면 약 333.33입니다.

=== 복구 후 (좋은 메타데이터) ===
🧠 step1 도구: calculator({'expr': '1000/3'}) → 👀 333.3333333333333

✅ [최종 답]
1000을 3으로 나누면 약 333.33입니다.


'1000을 3으로 나누면 약 333.33입니다.'

## STEP 7 — 병렬 호출 관찰: 독립 작업은 한 턴에 여러 도구

강의의 병렬 도구 호출 — 서로 독립적인 요청 2개를 한 문장에 담으면, 모델이 **한 턴에 tool_calls 여러 개**를 낼 수 있습니다.

In [27]:
# 검색(외부)과 계산(로컬)은 서로 독립 — 로그에서 한 스텝에 도구 2개가 찍히는지 관찰
run_agent("강남 매운 맛집을 찾아줘. 그리고 그것과 별개로 50000원을 4명이 나누면 1인당 얼마인지도 계산해줘.")
# 관찰: 순서 의존이 없으면 병렬 — 기다림은 가장 느린 도구만큼만

🧠 step1 도구: search_restaurants({'area': '강남', 'taste': '매운'}) → 👀 결과 없음
🧠 step1 도구: budget_check({'price': 50000, 'people': 4}) → 👀 1인당 12,500원

✅ [최종 답]
강남 지역에서 매운 맛집에 대한 결과는 없었습니다. 혹시 다른 지역이나 맛 종류로 검색해 보기를 원하시면 말씀해 주세요.

그리고 50,000원을 4명이 나누면 1인당 12,500원이 됩니다. 추가적인 도움이 필요하시면 언제든지 말씀해 주세요!


'강남 지역에서 매운 맛집에 대한 결과는 없었습니다. 혹시 다른 지역이나 맛 종류로 검색해 보기를 원하시면 말씀해 주세요.\n\n그리고 50,000원을 4명이 나누면 1인당 12,500원이 됩니다. 추가적인 도움이 필요하시면 언제든지 말씀해 주세요!'

## STEP 8 — 위험 도구에 승인 게이트 달기

'행동하는 도구'는 위험도 함께 옵니다. 삭제 도구를 추가하되, 실행 전 **사람 승인(Human-in-the-loop)** 을 강제합니다.

In [28]:
def delete_memo(name: str) -> str:
    ok = input(f"⚠ 파괴적 동작: 메모 '{name}' 삭제 — 승인? (y/n) ")
    if ok.strip().lower() != "y":
        return "사용자가 거부했습니다. 삭제하지 않았습니다."
    return f"'{name}' 메모를 삭제했습니다."

TOOL_IMPL["delete_memo"] = delete_memo
TOOLS.append({"type":"function","function":{
    "name":"delete_memo","description":"저장된 맛집 메모를 삭제한다(파괴적 — 신중히)",
    "parameters":{"type":"object","properties":{"name":{"type":"string"}},"required":["name"]}}})

run_agent("불막창 메모는 이제 필요 없어. 지워줘.", max_steps=3)
# 관찰: 읽기(기본 허용) — 쓰기(신중) — 파괴적(승인 필수)의 권한 3층

🧠 step1 도구: delete_memo({'name': '불막창'}) → 👀 사용자가 거부했습니다. 삭제하지 않았습니다.

✅ [최종 답]
메모 삭제에 실패했습니다. 필요할 경우 직접 삭제를 시도해보시기 바랍니다. 다른 도움을 드릴까요?


'메모 삭제에 실패했습니다. 필요할 경우 직접 삭제를 시도해보시기 바랍니다. 다른 도움을 드릴까요?'

## STEP 9 — ⭐ 내 팀 MVP에 적용: **필요한 MCP를 직접 찾아 연결**

오늘 배운 핵심은 *"모델이 약한 일은 도구로, 외부/공유가 필요한 도구는 MCP로"* 입니다.
이번엔 함수 도구를 손으로 짜는 대신, **우리 MVP에 필요한 능력을 MCP 서버로 직접 찾아 붙여**봅니다.

**진행 (4단계):**
1. **무엇이 필요한가** — 우리 MVP가 모델만으로 못 하는 외부 능력 1가지를 적는다 (예: 웹 검색, 문서 조회, 브라우저, 캘린더, DB, GitHub…).
2. **찾는다** — 그 능력을 주는 **MCP 서버를 직접 검색**한다.
   - 레지스트리/마켓플레이스: <https://github.com/modelcontextprotocol/servers> · <https://mcp.so> · 각 서비스 공식 문서("OOO MCP server")
   - ⌨️ Codex에게 시켜도 됨: `codex "우리 MVP엔 ____ 능력이 필요해. 적합한 MCP 서버 후보 2~3개와 codex mcp add 명령을 찾아줘"`
3. **연결한다** — 찾은 서버를 한 줄로 등록하고 권한을 확인한다.
   ```bash
   codex mcp add <별칭> -- npx -y <서버패키지>   # 예: codex mcp add context7 -- npx -y @upstash/context7-mcp
   codex mcp list                                # 등록 확인
   # Codex 세션 안에서  /mcp  로 연결·권한 점검
   ```
4. **검증한다** — `/mcp`로 도구가 보이면, 그 도구를 실제로 부르는 요청을 한 번 던져 결과를 확인한다.

> 고를 때 체크: **신뢰할 수 있는 출처인가 · 권한(읽기/쓰기/네트워크)이 과하지 않은가 · 우리 시나리오에 정말 필요한가(Less is More).**
> 파괴적 동작(쓰기·삭제)이 있는 MCP는 **사람 승인**을 거치도록.

In [31]:
# 팀별 작성: 우리 MVP에 '필요한 능력' → '찾은 MCP 서버' → '연결 명령'
# (함수로 만들 수 있는 로컬 도구는 STEP 2 형식으로, 외부/공유가 필요한 건 MCP로)
MY_MCP_PLAN = [
  # {"need":"최신 맛집 리뷰/영업시간 검색", "mcp":"context7", "add":"codex mcp add context7 -- npx -y @upstash/context7-mcp",
  #  "perms":"읽기 전용/네트워크", "why":"web_search 스텁을 실제 검색으로 교체"},
  # {"need":"...", "mcp":"...", "add":"codex mcp add ... -- ...", "perms":"...", "why":"..."},
]
for p in MY_MCP_PLAN:
    print(f"- 필요: {p['need']}")
    print(f"    → MCP: {p['mcp']}  (권한: {p.get('perms','?')})")
    print(f"    → 연결: {p['add']}")
    print(f"    → 이유: {p.get('why','')}\n")
if not MY_MCP_PLAN:
    print("⬜ 채워주세요 — 1) 필요한 외부 능력 1가지 → 2) MCP 레지스트리에서 서버 검색 →")
    print("   3) 터미널에서  codex mcp add  로 연결 →  4) /mcp 로 확인 후 실제 요청으로 검증")
    print("   (저녁 팀 프로젝트 Day2에서 이 MCP를 내 MVP 에이전트에 그대로 붙입니다)")

⬜ 채워주세요 — 1) 필요한 외부 능력 1가지 → 2) MCP 레지스트리에서 서버 검색 →
   3) 터미널에서  codex mcp add  로 연결 →  4) /mcp 로 확인 후 실제 요청으로 검증
   (저녁 팀 프로젝트 Day2에서 이 MCP를 내 MVP 에이전트에 그대로 붙입니다)


## 정리

- **도구 = 에이전트의 손발.** 정의→호출→실행의 한 사이클. 오늘은 7차시 루프에 **도구만 더했습니다**.
- **로컬(정밀)·API(실시간)** 두 종류, **적게·또렷이**(Less is More), **MCP=표준 연결**(`codex mcp add`).
- 안전: **권한은 최소로, 도구 출력은 데이터로**(프롬프트 인젝션 주의), 파괴적 동작엔 **사람 승인**.
- 다음 차시(10): 도구로 가져온 정보를 **기억하고 근거로 쓰는** 메모리 & RAG.
- 오늘 추가한 도구·MCP는 **저녁 팀 프로젝트(Day2)**에서 내 MVP에 바로 적용하세요.